In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s5e7/sample_submission.csv
/kaggle/input/playground-series-s5e7/train.csv
/kaggle/input/playground-series-s5e7/test.csv


In [2]:

#improt
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier


In [3]:
#load the files
train_df=pd.read_csv('/kaggle/input/playground-series-s5e7/train.csv')
test_df=pd.read_csv('/kaggle/input/playground-series-s5e7/test.csv')

#preview of dataset
train_df.head()

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,id,Time_spent_Alone,Stage_fear,Social_event_attendance,Going_outside,Drained_after_socializing,Friends_circle_size,Post_frequency,Personality
0,0,0.0,No,6.0,4.0,No,15.0,5.0,Extrovert
1,1,1.0,No,7.0,3.0,No,10.0,8.0,Extrovert
2,2,6.0,Yes,1.0,0.0,NaN,3.0,0.0,Introvert
3,3,3.0,No,7.0,3.0,No,11.0,5.0,Extrovert
4,4,1.0,No,4.0,4.0,No,13.0,NaN,Extrovert


In [4]:
# Check data types of all columns
print("Data Types:")
print(train_df.dtypes)
print("\n")

# Check for missing values
print("Missing Values:")
print(train_df.isnull().sum())
print("\n")


# Get basic statistics of numeric columns
print("Basic Statistics:")
print(train_df.describe())
print("\n")


Data Types:
id                             int64
Time_spent_Alone             float64
Stage_fear                    object
Social_event_attendance      float64
Going_outside                float64
Drained_after_socializing     object
Friends_circle_size          float64
Post_frequency               float64
Personality                   object
dtype: object


Missing Values:
id                              0
Time_spent_Alone             1190
Stage_fear                   1893
Social_event_attendance      1180
Going_outside                1466
Drained_after_socializing    1149
Friends_circle_size          1054
Post_frequency               1264
Personality                     0
dtype: int64


Basic Statistics:
                 id  Time_spent_Alone  Social_event_attendance  Going_outside  \
count  18524.000000      17334.000000             17344.000000   17058.000000   
mean    9261.500000          3.137764                 5.265106       4.044319   
std     5347.562529          3.003786    

In [5]:
df = train_df.copy()

num_imputer = SimpleImputer(strategy='mean')
cat_imputer = SimpleImputer(strategy='most_frequent')
cat_cols = ['Stage_fear', 'Drained_after_socializing']
target_col = 'Personality'
num_cols = ['Time_spent_Alone', 'Social_event_attendance', 'Going_outside', 
            'Friends_circle_size', 'Post_frequency']
df[num_cols] = num_imputer.fit_transform(df[num_cols])
df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

encoders = {}

# Encode input categorical columns
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

# Encode target column
target_encoder = LabelEncoder()
df[target_col] = target_encoder.fit_transform(df[target_col])
encoders[target_col] = target_encoder

train_df = df
train_df.head()

,id,Time_spent_Alone,Stage_fear,Social_event_attendance,Going_outside,Drained_after_socializing,Friends_circle_size,Post_frequency,Personality
0,0,0.0,0,6.0,4.0,0,15.0,5.000000,0
1,1,1.0,0,7.0,3.0,0,10.0,8.000000,0
2,2,6.0,1,1.0,0.0,0,3.0,0.000000,1
3,3,3.0,0,7.0,3.0,0,11.0,5.000000,0
4,4,1.0,0,4.0,4.0,0,13.0,4.982097,0


In [6]:
# Check data types of all columns
print("Data Types:")
print(train_df.dtypes)
print("\n")

# Check for missing values
print("Missing Values:")
print(train_df.isnull().sum())
print("\n")


# Get basic statistics of numeric columns
print("Basic Statistics:")
print(train_df.describe())
print("\n")


Data Types:
id                             int64
Time_spent_Alone             float64
Stage_fear                     int64
Social_event_attendance      float64
Going_outside                float64
Drained_after_socializing      int64
Friends_circle_size          float64
Post_frequency               float64
Personality                    int64
dtype: object


Missing Values:
id                           0
Time_spent_Alone             0
Stage_fear                   0
Social_event_attendance      0
Going_outside                0
Drained_after_socializing    0
Friends_circle_size          0
Post_frequency               0
Personality                  0
dtype: int64


Basic Statistics:
                 id  Time_spent_Alone    Stage_fear  Social_event_attendance  \
count  18524.000000      18524.000000  18524.000000             18524.000000   
mean    9261.500000          3.137764      0.217124                 5.265106   
std     5347.562529          2.905696      0.412299                 2.6

In [7]:
X = train_df.drop(columns=['id', 'Personality'])  # Features only
y = train_df['Personality']                       # Target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
cat = CatBoostClassifier(verbose=0)
lgb = LGBMClassifier(random_state=42)

xgb.fit(X_train, y_train)
cat.fit(X_train, y_train)
lgb.fit(X_train, y_train)

xgb_preds = xgb.predict_proba(X_test)
cat_preds = cat.predict_proba(X_test)
lgb_preds = lgb.predict_proba(X_test)

ensemble_preds_proba = (xgb_preds + cat_preds + lgb_preds) / 3
ensemble_preds = np.argmax(ensemble_preds_proba, axis=1)

y_true_df = pd.DataFrame({'target': y_test, 'id': X_test.index})
y_pred_df = pd.DataFrame({'target': ensemble_preds, 'id': X_test.index})

accuracy = accuracy_score(y_test, ensemble_preds)
print("Ensemble Accuracy:", accuracy)

[LightGBM] [Info] Number of positive: 3873, number of negative: 10946
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002534 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 67
[LightGBM] [Info] Number of data points in the train set: 14819, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.261354 -> initscore=-1.038945
[LightGBM] [Info] Start training from score -1.038945
Ensemble Accuracy: 0.9686909581646423


In [9]:
# Check data types of all columns
print("Data Types:")
print(test_df.dtypes)
print("\n")

# Check for missing values
print("Missing Values:")
print(test_df.isnull().sum())
print("\n")


# Get basic statistics of numeric columns
print("Basic Statistics:")
print(test_df.describe())
print("\n")


Data Types:
id                             int64
Time_spent_Alone             float64
Stage_fear                    object
Social_event_attendance      float64
Going_outside                float64
Drained_after_socializing     object
Friends_circle_size          float64
Post_frequency               float64
dtype: object


Missing Values:
id                             0
Time_spent_Alone             425
Stage_fear                   598
Social_event_attendance      397
Going_outside                466
Drained_after_socializing    432
Friends_circle_size          350
Post_frequency               408
dtype: int64


Basic Statistics:
                 id  Time_spent_Alone  Social_event_attendance  Going_outside  \
count   6175.000000       5750.000000              5778.000000    5709.000000   
mean   21611.000000          3.116870                 5.287989       4.037835   
std     1782.713288          2.985658                 2.758052       2.045207   
min    18524.000000          0.000000  

In [10]:
test_df_clean = test_df.copy()

test_df_clean[num_cols] = num_imputer.transform(test_df_clean[num_cols])
test_df_clean[cat_cols] = cat_imputer.transform(test_df_clean[cat_cols])

for col in cat_cols:
    le = encoders[col]
    test_df_clean[col] = le.transform(test_df_clean[col])
test_df_clean.head()

,id,Time_spent_Alone,Stage_fear,Social_event_attendance,Going_outside,Drained_after_socializing,Friends_circle_size,Post_frequency
0,18524,3.000000,0,7.0,4.0,0,6.0,4.982097
1,18525,3.137764,1,0.0,0.0,1,5.0,1.000000
2,18526,3.000000,0,5.0,6.0,0,15.0,9.000000
3,18527,3.000000,0,4.0,4.0,0,5.0,6.000000
4,18528,9.000000,1,1.0,2.0,1,1.0,1.000000


In [11]:
X_test_final = test_df_clean.drop(columns=['id'])

xgb_preds = xgb.predict_proba(X_test_final)
cat_preds = cat.predict_proba(X_test_final)

ensemble_proba = (xgb_preds + cat_preds) / 2
ensemble_preds = np.argmax(ensemble_proba, axis=1)

test_df_clean['Predicted_Personality'] = encoders[target_col].inverse_transform(ensemble_preds)
print(test_df_clean[['id', 'Predicted_Personality']])

         id Predicted_Personality
0     18524             Extrovert
1     18525             Introvert
2     18526             Extrovert
3     18527             Extrovert
4     18528             Introvert
...     ...                   ...
6170  24694             Extrovert
6171  24695             Introvert
6172  24696             Extrovert
6173  24697             Extrovert
6174  24698             Introvert

[6175 rows x 2 columns]


In [12]:
submission = test_df_clean[['id', 'Predicted_Personality']].copy()
submission.columns = ['id', 'Personality']
submission.to_csv('submission.csv', index=False)
from IPython.display import FileLink
FileLink('submission.csv')

/kaggle/working/submission.csv